In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
events_bronze = "/Volumes/main/lakehouse_marketing/bronze/events/"

df_events_bronze = spark.read\
                    .format("delta")\
                    .option("header", "true")\
                    .load(events_bronze)

display(df_events_bronze.limit(5))
display(df_events_bronze.printSchema())

* Selecionando as colunas

In [0]:
df = df_events_bronze.select(
                        "event_id",    
                        "user_id",
                        "campaign_id",
                        "event_type",
                        "event_timestamp",
                        "source_file",
                        "ingestion_timestamp"
)

##########################################################
# Aplicação / Reforço da Tipagem para (event_timestamp)
#########################################################
df_typed = df\
            .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))


##########################################################################
#Nomalização para a coluna event_type (Precisa aplicar upper + trim)
##########################################################################
df_normalized = df_typed.withColumn("event_type", F.upper(F.trim(F.col("event_type"))))


display(df_normalized.select("event_type").distinct())
display(df_normalized.count())


#### Regras de Transformação

* Check de nulos

In [0]:
# Verificando a quantidade de nulos na base
df_normalized.groupBy(F.col("user_id").isNull().alias("is_null")).count().show()

In [0]:
# # # REGRA 1: Remover eventos sem `user_id`
# df_valid = df_normalized.filter(F.col("user_id").isNotNull())

# # REGRA 2: Trazer apenas o conjunto de eventos permitidos
# allowed_events = ['VIEW', 'PURCHASE', 'CLICK']
 
# df_valid = df_valid.filter(F.upper(F.col("event_type")).isin(allowed_events))

# # REGRA 3: Trazer apenas `event_timestamp` validos
# df_valid = df_valid.filter(F.col("event_timestamp").isNotNull())
# display(df_valid)

Vamos aplicar 3 regras:
* REGRA 1: Remover eventos sem `user_id`
* REGRA 2: Trazer apenas o conjunto de eventos permitidos
* REGRA 3: Trazer apenas `event_timestamp` validos



In [0]:
# Define os eventos permitos, conforme a documentação
allowed_events = ['VIEW', 'PURCHASE', 'CLICK']


df_with_rules = df_normalized.withColumn(
    "rejection_reason",
    F.when(F.col("user_id").isNull(), "NULL_USER_ID")
     .when(~F.col("event_type").isin(allowed_events), "INVALID_EVENT_TYPE")
     .when(F.col("event_timestamp").isNull(), "NULL_EVENT_TIMESTAMP")
     .otherwise(None)
)

display(df_with_rules.sample(0.0001))

* Separar válidos e rejeitados

In [0]:
df_valid = df_with_rules.filter(F.col("rejection_reason").isNull())
df_rejected = df_with_rules.filter(F.col("rejection_reason").isNotNull())

display(df_valid.sample(0.0001))
display(df_rejected.sample(0.001))

#### Escrita na Silver

In [0]:
# Resultados válidos
df_valid.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable('main.silver_marketing.events')



# Registros rejeitados
df_rejected.write\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable('main.governance_marketing.events_rejected')


#### Validação Dados Silver - Events


In [0]:
%sql
SELECT * 
FROM main.silver_marketing.events TABLESAMPLE (0.01 PERCENT)


In [0]:
%sql
SELECT COUNT(*) AS n_registros
FROM main.silver_marketing.events